In [ ]:
import os
import psycopg2
import geopandas
import pandas as pd
from shapely.geometry import LineString

import warnings
warnings.simplefilter(action='ignore', category=UserWarning)

con = psycopg2.connect(database="time", user="postgres", host="localhost")

In [ ]:
# Read data
p1 = geopandas.read_postgis("select * from ilots_verniquet", con, geom_col='geom', index_col='gid')
p2 = geopandas.read_postgis("select * from ilots_vasserot", con, geom_col='geom', index_col='gid')
p3 = geopandas.read_postgis("select * from ilots_apur", con, geom_col='geom', index_col='gid')

In [ ]:
def match(geom1, geom2, threshold1, threshold2):
    if geom1.intersects(geom2):
        g1 = geom1.intersection(geom2)
        area = g1.area
        return area > threshold1/2.0 and area > threshold2 * min(geom1.area, geom2.area)
    return False

def match_tables(table1, table2, threshold):
    min_area = min(table1['geom'].apply(lambda x: x.area).min(),table2['geom'].apply(lambda x: x.area).min())
    newdata = pd.DataFrame(columns = ['id', 'id1', 'id2', 'geometry'])
    for index1, row1 in table1.iterrows():
        g1 = row1['geom']
        for index2, row2 in table2.iterrows():
            g2 = row2['geom']
            if match(g1, g2, min_area, threshold):
                line = LineString([g1.centroid,g2.centroid])
                newmatch = {'id':len(newdata), 'id1':index1, 'id2':index2, 'geometry':line}
                newdata.loc[len(newdata)] = newmatch
    return geopandas.GeoDataFrame(newdata, geometry='geometry')

In [ ]:
ilots_stables_verniquet_vasserot = match_tables(p1, p2, 0.2)
ilots_stables_verniquet_vasserot.set_crs(p1.crs, inplace=True)

ilots_stables_vasserot_apur = match_tables(p1, p3, 0.2)
ilots_stables_vasserot_apur.set_crs(p1.crs, inplace=True)

ilots_stables_verniquet_apur = match_tables(p2, p3, 0.2)
ilots_stables_verniquet_apur.set_crs(p1.crs, inplace=True)

In [ ]:
len(ilots_stables_verniquet_vasserot), len(ilots_stables_vasserot_apur), len(ilots_stables_verniquet_apur)

In [ ]:
# Select distinct matches
ilots_stables_verniquet_vasserot.drop_duplicates(subset='id1', inplace=True)
ilots_stables_verniquet_vasserot.drop_duplicates(subset='id2', inplace=True)

ilots_stables_vasserot_apur.drop_duplicates(subset='id1', inplace=True)
ilots_stables_vasserot_apur.drop_duplicates(subset='id2', inplace=True)

ilots_stables_verniquet_apur.drop_duplicates(subset='id1', inplace=True)
ilots_stables_verniquet_apur.drop_duplicates(subset='id2', inplace=True)

In [ ]:
len(ilots_stables_verniquet_vasserot), len(ilots_stables_vasserot_apur), len(ilots_stables_verniquet_apur)

In [ ]:
# Save data to geojson
ilots_stables_verniquet_vasserot.to_file("../data/ilots/ilots_stables_verniquet_vasserot.geojson", driver='GeoJSON')
ilots_stables_vasserot_apur.to_file("../data/ilots/ilots_stables_vasserot_apur.geojson", driver='GeoJSON')
ilots_stables_verniquet_apur.to_file("../data/ilots/ilots_stables_verniquet_apur.geojson", driver='GeoJSON')